# FF3 선정 결과의 선형성·꼬리 진단 (2000-01 ~ 2010-12)

**목적**: `ff3_selection_2000_2010` 이 고른 131종목의 t(α) 가 (a) 비선형 시장 반응, (b) 비정규 잔차, (c) 소수의 극단적인 달 때문에 생긴 것인지 점검한다. 선정 자체는 바꾸지 않는다.

**진단 항목**
1. Treynor-Mazuy: FF3 + (Mkt−RF)² 항, 제곱항 |t| > 2 인 비율
2. Henriksson-Merton: FF3 + max(0, −(Mkt−RF)) 항, 하락장 항 |t| > 2 인 비율
3. 잔차 첨도(정규분포 = 3) 중앙값, Jarque-Bera 정규성 기각(p < 0.05) 비율
4. 잔차 부트스트랩: α = 0 귀무하에서 종목당 500회 재추정한 t(α) 의 단측 95% 분위수 — **난수 시드 0 으로 고정**, 종목은 PERMNO 오름차순으로 처리하므로 재실행해도 같은 값(1.624)이 나온다. 시드를 바꾸면 중앙값이 약 1.60~1.62 범위에서 달라진다.
5. |Mkt−RF| 가 가장 큰 5개월 제외 후 α > 0, t(α) ≥ 2 유지 비율
6. 시장 초과수익률 5분위별 선정 종목 평균 잔차


In [1]:
import os                                             # 경로 확인·폴더 생성
import zipfile                                        # 팩터 파일이 zip 으로만 있을 때 직접 열기
import numpy as np                                    # 행렬 연산 (OLS)
import pandas as pd                                   # 데이터프레임 처리
from scipy import stats                               # 첨도·Jarque-Bera·t 분포

## 0. 경로 및 파라미터 (유니버스 규칙은 선정 코드와 동일)

In [2]:

BASE = (os.path.dirname(os.path.abspath(__file__))    # .py 로 실행하면 스크립트가 있는 폴더
        if "__file__" in globals() else os.getcwd())  # 노트북이면 현재 작업 폴더
DATA_DIR = os.path.join(BASE, "data")                 # 입력 파일 폴더
OUTPUT_DIR = os.path.join(BASE, "outputs")            # 결과 폴더
SELECTED_NAME = "ff3_selected_2000~2010.csv"          # 입력으로 받을 선정 결과 파일 이름
SUMMARY_FILE = os.path.join(OUTPUT_DIR, "ff3_diagnostics_summary.csv")  # 이 코드의 산출물

SAMPLE_START = pd.Period("2000-01", freq="M")         # 분석 시작월
SAMPLE_END = pd.Period("2010-12", freq="M")           # 분석 종료월 (이후 데이터는 사용 안 함)
VALID_EXCHCD = [1, 2, 3]                              # 허용 거래소: 1=NYSE, 2=AMEX, 3=NASDAQ
RET_CAP = 5.0                                         # 월수익률 +500% 초과는 결측 처리 (선정 코드와 동일)
TRACKING_STOCK_NAMES = {"APPLERA CORP", "LOEWS CORP"} # 트래킹 주식은 회사명이 같아도 합치지 않음

T_CUT = 2.0                                           # 유의성 기준 |t| (약 5% 양측)
JB_ALPHA = 0.05                                       # Jarque-Bera 유의수준
N_BOOT = 500                                          # 부트스트랩 반복 횟수 (종목당)
BOOT_SEED = 0                                         # 부트스트랩 난수 시드: 0 으로 고정 (바꾸면 임계값 중앙값이 약 1.60~1.62 범위에서 달라짐)
N_EXTREME = 5                                         # 제외할 극단월 개수
N_QUANTILE = 5                                        # 시장 수익률 분위 개수 (5분위)

## 보조 함수

In [3]:
def find_input(path_candidates):                      # 후보 경로 중 실제 존재하는 첫 경로를 찾는다
    for path in path_candidates:                      # 후보를 순서대로 확인
        if os.path.exists(path):                      # 파일이 있으면
            return path                               # 그 경로를 돌려준다
    raise FileNotFoundError(f"{path_candidates} 중 존재하는 파일이 없음")  # 끝까지 못 찾으면 중단


def read_ff_text(path):                               # 팩터 파일 원문을 문자열로 읽는다
    if path.lower().endswith(".zip"):                 # zip 파일이면
        with zipfile.ZipFile(path) as zf:             # 압축을 열고
            inner = [n for n in zf.namelist()         # 안에 든 파일 중
                     if n.lower().endswith(".csv")][0]  # 첫 번째 CSV 를 골라
            return zf.read(inner).decode("latin-1")   # 바이트를 문자열로 바꿔 반환
    with open(path, encoding="latin-1") as f:         # 일반 CSV 면
        return f.read()                               # 그대로 읽어 반환


def load_ff3(path):                                   # French 원본 포맷 → 월별 팩터 DataFrame
    rows = []                                         # 월별 행을 모을 리스트
    for ln in read_ff_text(path).splitlines():        # 파일을 한 줄씩 순회
        p = [x.strip() for x in ln.split(",")]        # 쉼표로 나누고 공백 제거
        if len(p) == 5 and p[0].isdigit() and len(p[0]) == 6:  # YYYYMM 으로 시작하는 월별 행만
            rows.append(p)                            # 채택
    ff = pd.DataFrame(rows, columns=["ym", "MKT_RF", "SMB", "HML", "RF"])  # 열 이름 부여
    ff["ym"] = pd.PeriodIndex(ff["ym"].str[:4] + "-" + ff["ym"].str[4:], freq="M")  # 'YYYYMM' → 월 Period
    return ff.set_index("ym").astype(float) / 100.0   # 월을 인덱스로, % 단위를 소수로


def ols(y, X):                                        # 절편 포함 OLS: 계수, t-통계량, 잔차 반환
    Xc = np.column_stack([np.ones(len(y)), X])        # 절편 열을 붙인 설명변수
    XtX_inv = np.linalg.inv(Xc.T @ Xc)                # (X'X)^-1
    b = XtX_inv @ Xc.T @ y                            # OLS 계수
    e = y - Xc @ b                                    # 잔차
    s2 = (e @ e) / (len(y) - Xc.shape[1])             # 잔차분산 (자유도 = 관측 수 - 계수 수)
    t = b / np.sqrt(np.diag(XtX_inv) * s2)            # 계수별 t-통계량
    return b, t, e                                    # 계수, t, 잔차

## 1. 데이터 로드 — 2010-12 이후는 즉시 삭제 (선정 코드와 동일)

In [4]:
PATH_STOCK = find_input([os.path.join(DATA_DIR, "IFE_term_stock_data.csv"),  # data/ 폴더 우선
                         os.path.join(BASE, "IFE_term_stock_data.csv")])     # 없으면 BASE 바로 아래
raw = pd.read_csv(PATH_STOCK, parse_dates=["date"])   # 주가 패널 읽기
raw["ym"] = raw["date"].dt.to_period("M")             # 날짜 → 월 Period
raw = raw[(raw["ym"] >= SAMPLE_START) & (raw["ym"] <= SAMPLE_END)].copy()  # 2000-01~2010-12 만 남김
assert raw["ym"].max() <= SAMPLE_END                  # 2011년 이후 행이 없는지 확인
raw = raw.drop_duplicates(subset=["PERMNO", "ym"], keep="first")  # 같은 종목·같은 월 중복 제거
raw = raw[raw["ADJ_PRC"] > 0]                         # 가격 결측·0 이하 행 제거
raw = raw.sort_values(["PERMNO", "ym"]).reset_index(drop=True)  # 종목·월 순으로 정렬
raw["mnum"] = raw["ym"].dt.year * 12 + raw["ym"].dt.month  # 월을 정수로 바꿈 (연속월 판정용)

PATH_FF = find_input([os.path.join(d, n) for d in (DATA_DIR, BASE)  # 팩터 파일 후보 경로
                      for n in ("F-F_Research_Data_Factors.csv",   # 압축 푼 파일 (소문자 확장자)
                                "F-F_Research_Data_Factors.CSV",   # 압축 푼 파일 (대문자 확장자)
                                "F-F_Research_Data_Factors_CSV.zip")])  # 압축 그대로
ff3 = load_ff3(PATH_FF).loc[SAMPLE_START:SAMPLE_END]  # 팩터를 읽고 분석 구간만 남김
assert ff3.index.max() <= SAMPLE_END                  # 2011년 이후 팩터가 없는지 확인

SELECTED_FILE = find_input([os.path.join(d, SELECTED_NAME)  # 선정 결과 파일 후보 경로
                             for d in (DATA_DIR, BASE, OUTPUT_DIR)])  # data/ → 기본 폴더 → outputs/ 순
sel = pd.read_csv(SELECTED_FILE, encoding="utf-8-sig")  # 선정 결과 읽기 (BOM 이 있어도 열 이름이 깨지지 않게)
assert "PERMNO" in sel.columns, "선정 파일에 PERMNO 열이 없음"  # 필수 열 확인
assert sel["PERMNO"].is_unique, "선정 파일에 중복 PERMNO 가 있음"  # 중복 종목 확인
print(f"[0] 선정 결과: {os.path.basename(SELECTED_FILE)} ({len(sel)}종목)")  # 읽은 파일과 종목 수 출력
sel_permno = set(sel["PERMNO"].astype(int))           # 선정 종목 PERMNO 집합

[0] 선정 결과: ff3_selected_2000~2010.csv (131종목)


## 2. 유니버스 재구성 (balanced panel, 선정 코드와 같은 규칙)

In [5]:
n_months = len(pd.period_range(SAMPLE_START, SAMPLE_END, freq="M"))  # 132개월
n_rets = n_months - 1                                 # 월수익률 131개
price_cnt = raw.groupby("PERMNO")["ym"].nunique()     # 종목별 주가가 있는 월 수
sub = raw[raw["PERMNO"].isin(price_cnt[price_cnt == n_months].index)].copy()  # 132개월 주가 보유 종목
grp = sub.groupby("PERMNO")                           # 종목별로 묶음
gap = sub["mnum"] - grp["mnum"].shift(1)              # 직전 관측과의 월 간격
sub["RET"] = np.where(gap == 1, sub["ADJ_PRC"] / grp["ADJ_PRC"].shift(1) - 1.0, np.nan)  # 연속월일 때만 수익률
sub.loc[sub["RET"] > RET_CAP, "RET"] = np.nan         # +500% 초과는 결측 (선정 코드와 동일)
ret_cnt = sub.dropna(subset=["RET"]).groupby("PERMNO").size()  # 종목별 유효 수익률 개수
balanced = ret_cnt[ret_cnt == n_rets].index           # 131개 모두 유효한 종목
at_end = sub[sub["ym"] == SAMPLE_END].set_index("PERMNO")  # 2010-12 행
balanced = balanced.intersection(at_end.index[at_end["EXCHCD"].isin(VALID_EXCHCD)])  # 2010-12 거래소 조건
panel = sub[sub["PERMNO"].isin(balanced)].dropna(subset=["RET"]).copy()  # 분석용 수익률 패널
ident = at_end.loc[balanced, ["COMNAM"]].copy()       # 회사명 (복수 클래스 정리용)
ident["avg_vol"] = panel.groupby("PERMNO")["VOL"].mean()  # 구간 평균 거래량
is_track = ident["COMNAM"].isin(TRACKING_STOCK_NAMES)  # 트래킹 주식 여부
dups = ident[~is_track & ident["COMNAM"].duplicated(keep=False)]  # 회사명이 겹치는 복수 클래스
dups = dups.sort_values(["COMNAM", "avg_vol"], ascending=[True, False])  # 회사별 거래량 큰 순
drop = dups.index.difference(dups.drop_duplicates("COMNAM", keep="first").index)  # 거래량 1위 외 클래스
panel = panel[~panel["PERMNO"].isin(drop)]            # 복수 클래스 정리

R = panel.pivot(index="ym", columns="PERMNO", values="RET")  # 월 × 종목 수익률 행렬
assert R.notna().all().all() and len(R) == n_rets    # 결측 없는 131개월인지 확인
assert sel_permno.issubset(set(R.columns))            # 선정 종목이 모두 유니버스에 있는지 확인
F = ff3.loc[R.index, ["MKT_RF", "SMB", "HML"]].to_numpy()  # 팩터 행렬 (T×3)
Y = R.to_numpy() - ff3.loc[R.index, "RF"].to_numpy()[:, None]  # 초과수익률 (T×N)
mkt = F[:, 0]                                         # 시장 초과수익률 (Mkt-RF)
T, N = Y.shape                                        # 관측 월 수, 종목 수
is_sel = np.array([p in sel_permno for p in R.columns])  # 선정 종목 여부 (열 순서)
print(f"[1] 유니버스 {N:,}종목, 선정 {is_sel.sum()}종목, {T}개월")  # 진행 상황 출력

[1] 유니버스 2,426종목, 선정 131종목, 131개월


## 3. 진단 1~3: Treynor-Mazuy, Henriksson-Merton, 첨도·Jarque-Bera (전 종목)

In [6]:
X_tm = np.column_stack([F, mkt ** 2])                 # FF3 + 시장 제곱항
X_hm = np.column_stack([F, np.maximum(0.0, -mkt)])    # FF3 + 하락장 항 max(0, -(Mkt-RF))
tm_sig = np.zeros(N, bool)                            # 제곱항 유의 여부
hm_sig = np.zeros(N, bool)                            # 하락장 항 유의 여부
kurt = np.zeros(N)                                    # FF3 잔차 첨도 (정규분포 = 3)
jb_rej = np.zeros(N, bool)                            # Jarque-Bera 정규성 기각 여부
E = np.zeros((T, N))                                  # FF3 잔차 저장 (5분위 분석용)
for j in range(N):                                    # 종목마다
    y = Y[:, j]                                       # 해당 종목 초과수익률
    _, _, e = ols(y, F)                               # 기본 FF3 잔차
    E[:, j] = e                                       # 잔차 저장
    tm_sig[j] = abs(ols(y, X_tm)[1][4]) > T_CUT       # 제곱항 t 가 기준을 넘는지
    hm_sig[j] = abs(ols(y, X_hm)[1][4]) > T_CUT       # 하락장 항 t 가 기준을 넘는지
    kurt[j] = stats.kurtosis(e, fisher=False)         # 첨도 (Pearson 정의, 정규분포 = 3)
    jb_rej[j] = stats.jarque_bera(e).pvalue < JB_ALPHA  # 정규성 기각 여부

## 4. 진단 4: 잔차 부트스트랩 임계값 (선정 종목)

In [7]:
rng = np.random.default_rng(BOOT_SEED)                # 시드 0 으로 고정한 난수 생성기 (numpy PCG64)
Xc = np.column_stack([np.ones(T), F])                 # 절편 포함 설명변수
boot_crit = []                                        # 종목별 부트스트랩 5% 임계값
assert list(R.columns) == sorted(R.columns)           # 종목 순서가 PERMNO 오름차순인지 확인 (난수 배정 순서 고정)
for j in np.where(is_sel)[0]:                         # 선정 종목마다 (PERMNO 오름차순)
    b, _, e = ols(Y[:, j], F)                         # FF3 추정
    fit_null = Xc @ np.r_[0.0, b[1:]]                 # α = 0 으로 둔 적합값 (귀무가설)
    t_boot = np.empty(N_BOOT)                         # 부트스트랩 t(α) 저장
    for k in range(N_BOOT):                           # 반복마다
        y_star = fit_null + e[rng.integers(0, T, T)]  # 잔차를 복원추출해 가짜 수익률 생성
        t_boot[k] = ols(y_star, F)[1][0]              # 그때의 t(α)
    boot_crit.append(np.quantile(t_boot, 0.95))       # 단측 95% 분위수 = 5% 임계값
t_crit = stats.t.ppf(0.95, T - 4)                     # 비교용 t 분포 임계값 (자유도 127)

## 5. 진단 5: 극단월 제외 후 재추정 (선정 종목)

In [8]:
ext_idx = np.argsort(np.abs(mkt))[-N_EXTREME:]        # |Mkt-RF| 가 가장 큰 5개월 위치
keep = np.ones(T, bool)                               # 사용할 월 표시
keep[ext_idx] = False                                 # 극단월 제외
survive = 0                                           # 기준 유지 종목 수
for j in np.where(is_sel)[0]:                         # 선정 종목마다
    b, t, _ = ols(Y[keep, j], F[keep])                # 극단월 뺀 FF3
    survive += int(b[0] > 0 and t[0] >= T_CUT)        # α > 0, t(α) ≥ 2 유지 여부
ext_months = [str(R.index[i]) for i in sorted(ext_idx)]  # 제외한 월 이름

## 6. 시장 초과수익률 5분위별 평균 잔차

In [9]:
q = pd.qcut(mkt, N_QUANTILE, labels=False)            # 월별 시장 수익률 분위 (0 = 최하위)
resid_sel = E[:, is_sel].mean(axis=1)                 # 월별 선정 종목 평균 잔차
resid_all = E.mean(axis=1)                            # 월별 유니버스 평균 잔차

## 7. 요약 출력·저장

In [10]:
pct = lambda m: round(100 * m.mean(), 1)              # 비율을 % 로 바꾸는 함수
rows = [                                              # 항목, 선정 종목 값, 유니버스 값, 단위
    ("Treynor-Mazuy 제곱항 유의 비율", pct(tm_sig[is_sel]), pct(tm_sig), "%"),  # 진단 1
    ("Henriksson-Merton 하락장 항 유의 비율", pct(hm_sig[is_sel]), pct(hm_sig), "%"),  # 진단 2
    ("잔차 첨도 중앙값 (정규=3)", round(float(np.median(kurt[is_sel])), 2), round(float(np.median(kurt)), 2), "배수"),  # 진단 3-a
    ("Jarque-Bera 기각 비율", pct(jb_rej[is_sel]), pct(jb_rej), "%"),  # 진단 3-b
    (f"부트스트랩 5% 임계값 중앙값 (시드 {BOOT_SEED})", round(float(np.median(boot_crit)), 3), round(float(t_crit), 3), "t (유니버스 열 = t 분포 값)"),  # 진단 4 (유니버스 열에 t 분포 임계값)
    ("극단 5개월 제외 후 기준 유지", round(100 * survive / is_sel.sum(), 1), np.nan, f"% ({survive}/{is_sel.sum()}종목)"),  # 진단 5 (유니버스 값 없음)
    ("시장 하위 20% 달 평균 잔차", round(100 * resid_sel[q == 0].mean(), 2), round(100 * resid_all[q == 0].mean(), 2), "%/월"),  # 하락장 잔차
    ("시장 상위 20% 달 평균 잔차", round(100 * resid_sel[q == N_QUANTILE - 1].mean(), 2), round(100 * resid_all[q == N_QUANTILE - 1].mean(), 2), "%/월"),  # 상승장 잔차
    ("시장 중간 60% 달 평균 잔차", round(100 * resid_sel[(q > 0) & (q < N_QUANTILE - 1)].mean(), 2), round(100 * resid_all[(q > 0) & (q < N_QUANTILE - 1)].mean(), 2), "%/월"),  # 중간 구간 잔차
]                                                     # 요약 표 끝
summary = pd.DataFrame(rows, columns=["항목", "선정", "유니버스", "단위"])  # 데이터프레임으로
print(summary.to_string(index=False))                 # 화면 출력
print(f"제외한 극단월: {ext_months}")                  # 극단월 목록 출력
print(f"시장 하위 20% 달: {(q == 0).sum()}개월, 평균 Mkt-RF {100 * mkt[q == 0].mean():.2f}%")  # 하위 분위 정보
os.makedirs(OUTPUT_DIR, exist_ok=True)                # 결과 폴더 생성
summary.to_csv(SUMMARY_FILE, index=False, encoding="utf-8-sig")  # 요약 저장 (엑셀 호환 BOM)
print(f"[2] 저장: {SUMMARY_FILE}")                     # 저장 위치 출력

                           항목     선정   유니버스                  단위
      Treynor-Mazuy 제곱항 유의 비율  2.300  8.000                   %
Henriksson-Merton 하락장 항 유의 비율  1.500  6.800                   %
             잔차 첨도 중앙값 (정규=3)  4.170  5.360                  배수
            Jarque-Bera 기각 비율 62.600 83.200                   %
      부트스트랩 5% 임계값 중앙값 (시드 0)  1.624  1.657 t (유니버스 열 = t 분포 값)
            극단 5개월 제외 후 기준 유지 87.800    NaN       % (115/131종목)
            시장 하위 20% 달 평균 잔차  0.360  0.230                 %/월
            시장 상위 20% 달 평균 잔차 -0.110  0.060                 %/월
            시장 중간 60% 달 평균 잔차 -0.090 -0.100                 %/월
제외한 극단월: ['2000-11', '2002-09', '2008-10', '2009-02', '2009-04']
시장 하위 20% 달: 27개월, 평균 Mkt-RF -7.34%
[2] 저장: /content/outputs/ff3_diagnostics_summary.csv
